Connected to Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence, ensure_edge_index_column_sorted
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    if len(sys.argv) >= 3:
        datasets = [sys.argv[1]]  # HPC: dataset via submit.sh arg
        models = [sys.argv[2]]    # HPC: model via submit.sh arg
    else:
        datasets = ["IBM_AML_LiSmall"]
        models = ["MLP","GIN"]
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["AMLSim"]
    models = ["SVM", "RF", "XGB", "MLP"]
#"IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "AMLSim", "Elliptic"
print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")
        data = ensure_edge_index_column_sorted(data, name=dataset)

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 15.4 GB available, 50.4% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 1 datasets: AMLSim

Hyperparameter tuning for datasets 1/1: AMLSim



c:\Users\lambe\Desktop\Final_script\pre_processing.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Study 'SVM_optimization on AMLSim dataset' not found.
Dataset AMLSim loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for AMLSim dataset...
Data kept on CPU for NeighborLoader-based training (AMLSim)
Starting hyperparameter optimization for AMLSim dataset...
Starting hyperparameter tuning for dataset: AMLSim


Models:   0%|          | 0/4 [00:00<?, ?model/s]

...

Best val PR-AUC for RF on AMLSim: 0.1625
Study 'XGB_optimization on AMLSim dataset' not found.
Starting optimization for XGB on AMLSim with 100 trials.


[I 2026-06-01 20:24:31,487] Trial 0 finished with value: 0.14710357436985522 and parameters: {'max_depth': 11, 'Gamma_XGB': 0.7025210716314528, 'n_estimators': 450, 'learning_rate_XGB': 0.06290938557134781, 'colsample_bytree': 0.7254631092305031, 'subsample': 0.8713348867819091, 'min_child_weight': 9, 'reg_alpha': 2.5672718408900045e-06, 'reg_lambda': 3.335592275406104}. Best is trial 0 with value: 0.14710357436985522.
[I 2026-06-01 20:24:32,035] Trial 1 finished with value: 0.14751202852381207 and parameters: {'max_depth': 12, 'Gamma_XGB': 0.8237699845448265, 'n_estimators': 400, 'learning_rate_XGB': 0.019765590235572835, 'colsample_bytree': 0.9124336320568585, 'subsample': 0.5955756729749178, 'min_child_weight': 3, 'reg_alpha': 1.2889871046977888e-06, 'reg_lambda': 1.423441266423325e-06}. Best is trial 1 with value: 0.14751202852381207.
[I 2026-06-01 20:24:32,520] Trial 2 finished with value: 0.1490651242509991 and parameters: {'max_depth': 11, 'Gamma_XGB': 0.7263871718630316, 'n_est

Best val PR-AUC for XGB on AMLSim: 0.1639
Study 'MLP_optimization on AMLSim dataset' not found.
Starting optimization for MLP on AMLSim with 150 trials.


[I 2026-06-01 20:25:20,330] Trial 0 finished with value: 0.16495739055270403 and parameters: {'hidden_units': 65, 'dropout_1': 0.5286507291463909, 'dropout_2': 0.33853296510060027, 'learning_rate': 0.001498259698376278, 'weight_decay': 1.5033663714133595e-05, 'gamma_focal': 1.4662111317378963, 'n_epochs': 196}. Best is trial 0 with value: 0.16495739055270403.
[I 2026-06-01 20:25:25,696] Trial 1 finished with value: 0.17694609559989627 and parameters: {'hidden_units': 237, 'dropout_1': 0.022418338591820162, 'dropout_2': 0.4519612099529556, 'learning_rate': 0.003202852838275819, 'weight_decay': 0.0005876669393941697, 'gamma_focal': 3.1648340391391625, 'n_epochs': 175}. Best is trial 1 with value: 0.17694609559989627.
[I 2026-06-01 20:25:26,845] Trial 2 finished with value: 0.17865707767574812 and parameters: {'hidden_units': 152, 'dropout_1': 0.2556936632525506, 'dropout_2': 0.26862233638083904, 'learning_rate': 0.010193568582568154, 'weight_decay': 0.0013760418049796948, 'gamma_focal': 

Best val PR-AUC for MLP on AMLSim: 0.2167
GPU memory cleared after AMLSim

✓ Successfully completed tuning for dataset 1/1: AMLSim


HYPERPARAMETER TUNING COMPLETE
All 1 datasets have been processed.


Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence, ensure_edge_index_column_sorted
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    if len(sys.argv) >= 3:
        datasets = [sys.argv[1]]  # HPC: dataset via submit.sh arg
        models = [sys.argv[2]]    # HPC: model via submit.sh arg
    else:
        datasets = ["IBM_AML_LiSmall"]
        models = ["MLP","GIN"]
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["AMLSim"]
    models = ["SVM", "RF", "XGB", "MLP", "GCN", "GAT", "GIN"]
#"IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "AMLSim", "Elliptic"
print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")
        data = ensure_edge_index_column_sorted(data, name=dataset)

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 16.1 GB available, 48.3% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 1 datasets: AMLSim

Hyperparameter tuning for datasets 1/1: AMLSim

Study 'SVM_optimization on AMLSim dataset' complete: 100/35 trials done.
Study 'RF_optimization on AMLSim dataset' complete: 100/35 trials done.
Study 'XGB_optimization on AMLSim dataset' complete: 100/35 trials don

c:\Users\lambe\Desktop\Final_script\pre_processing.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Study 'MLP_optimization on AMLSim dataset' complete: 150/150 trials done.
Study 'GCN_optimization on AMLSim dataset' not found.
Dataset AMLSim loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for AMLSim dataset...
Data kept on CPU for NeighborLoader-based training (AMLSim)
Starting hyperparameter optimization for AMLSim dataset...
Starting hyperparameter tuning for dataset: AMLSim


Models:  43%|████▎     | 3/7 [00:00<00:00, 23.46model/s]

...

Best val PR-AUC for GAT on AMLSim: 0.1896
Study 'GIN_optimization on AMLSim dataset' not found.
Searching for optimal tuning batch size...
Running in TUNING mode: will use up to 70% of GPU memory
GPU memory limit: 11.19 GB
Fallback testing batch size: 16384


[I 2026-06-01 21:14:53,807] A new study created in RDB with name: GIN_optimization on AMLSim dataset


Optimal tuning batch size found: 15564
Saved tuning batch size 15564 for AMLSim_GIN
Starting optimization for GIN on AMLSim with 150 trials.


[I 2026-06-01 21:14:54,496] Trial 0 finished with value: 0.16335409200173057 and parameters: {'hidden_units': 64, 'dropout': 0.0002863042745641087, 'n_layers': 1, 'learning_rate': 0.02025660905865202, 'weight_decay': 0.0009569022150244828, 'gamma_focal': 4.437639608460758, 'n_epochs': 7}. Best is trial 0 with value: 0.16335409200173057.
[I 2026-06-01 21:15:17,261] Trial 1 finished with value: 0.1946458515632502 and parameters: {'hidden_units': 236, 'dropout': 0.21890895484143058, 'n_layers': 2, 'learning_rate': 0.005422658360899864, 'weight_decay': 0.0007794056314269695, 'gamma_focal': 2.6112110455497346, 'n_epochs': 209}. Best is trial 1 with value: 0.1946458515632502.
[I 2026-06-01 21:15:25,274] Trial 2 finished with value: 0.17292990978248501 and parameters: {'hidden_units': 146, 'dropout': 0.2831423190319477, 'n_layers': 1, 'learning_rate': 0.017432715541113853, 'weight_decay': 0.003069174224483346, 'gamma_focal': 3.1459977950500946, 'n_epochs': 65}. Best is trial 1 with value: 0.1

Best val PR-AUC for GIN on AMLSim: 0.2038
GPU memory cleared after AMLSim

✓ Successfully completed tuning for dataset 1/1: AMLSim


HYPERPARAMETER TUNING COMPLETE
All 1 datasets have been processed.


In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data = ensure_edge_index_column_sorted(data, name=dataset)
    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")


Final evaluation for dataset 1/1: AMLSim

Loaded best params for SVM (best F1-illicit: 0.1534)
Loaded best params for RF (best F1-illicit: 0.1625)
Loaded best params for XGB (best F1-illicit: 0.1639)
Loaded best params for MLP (best F1-illicit: 0.2167)
Loaded best params for GCN (best F1-illicit: 0.1948)
Loaded best params for GAT (best F1-illicit: 0.1896)
Loaded best params for GIN (best F1-illicit: 0.2038)


c:\Users\lambe\Desktop\Final_script\pre_processing.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Data kept on CPU for NeighborLoader-based evaluation (AMLSim)

Starting FINAL EVALUATION for AMLSim dataset...
Evaluating SVM on AMLSim...
  > Using FULL BATCH for SVM on AMLSim (no NeighborLoader)
  > Run 1/10 (Seed 3785470329)
Saved metrics to results/AMLSim/pkl_files/SVM_run_1_pr_data_local-14076.pkl
Saved metrics to results/AMLSim/pkl_files/SVM_run_1_metrics_local-14076.pkl
  > Run 2/10 (Seed 3335016359)
Saved metrics to results/AMLSim/pkl_files/SVM_run_2_pr_data_local-14076.pkl
Saved metrics to results/AMLSim/pkl_files/SVM_run_2_metrics_local-14076.pkl
  > Run 3/10 (Seed 2378348718)
Saved metrics to results/AMLSim/pkl_files/SVM_run_3_pr_data_local-14076.pkl
Saved metrics to results/AMLSim/pkl_files/SVM_run_3_metrics_local-14076.pkl
  > Run 4/10 (Seed 2237500742)
Saved metrics to results/AMLSim/pkl_files/SVM_run_4_pr_data_local-14076.pkl
Saved metrics to results/AMLSim/pkl_files/SVM_run_4_metrics_local-14076.pkl
  > Run 5/10 (Seed 2068858612)
Saved metrics to results/AMLSim/pkl_fil